In [ ]:
# ============================================
# Kria DPU RF Classifier Test (no TensorFlow)
# - uses pynq_dpu runner + numpy
# - computes accuracy, confusion matrix, Acc vs SNR
# ============================================
import os, time, math
import numpy as np
import matplotlib.pyplot as plt

# Optional (recommended) for confusion matrix only:
try:
    from sklearn.metrics import confusion_matrix
    HAVE_SK = True
except Exception:
    HAVE_SK = False
    print("sklearn not found: confusion matrix will be skipped.")

# ---------- User paths & params ----------
BIT_PATH     = "/home/xilinx/dpu.bit"                     # or wherever your bitstream lives
XMODEL_PATH  = "/workspace/vai_c_output/rf_F32_t1.xmodel" # compiled model for DPU
RF_INPUT_PATH   = "/workspace/rf_input.npy"               # (N, 1024, 1, 2) or similar
RF_CLASSES_PATH = "/workspace/rf_classes.npy"             # optional: (N,) ints
RF_SNRS_PATH    = "/workspace/rf_snrs.npy"                # optional: (N,) floats/ints
NUM_FRAMES      = 256                                     # None -> all
BATCH_SIZE      = 32
MODS = ['BPSK','QPSK','GMSK','FM','OOK','OQPSK','8PSK','16QAM','AM-SSB-WC','AM-DSB-SC']  # adjust if needed

# ---------- DPU init ----------
from pynq_dpu import DpuOverlay
overlay = DpuOverlay(BIT_PATH)
overlay.load_model(XMODEL_PATH)
dpu = overlay.runner

in_tensors  = dpu.get_input_tensors()
out_tensors = dpu.get_output_tensors()
assert len(in_tensors) == 1 and len(out_tensors) == 1, "This script assumes a single-input, single-output network."

in_shape  = tuple(in_tensors[0].dims)   # e.g., (batch, H, W, C) OR (batch, C, H, W)
out_shape = tuple(out_tensors[0].dims)  # e.g., (batch, num_classes)
print("DPU input shape:", in_shape)
print("DPU output shape:", out_shape)

# Heuristics for layout detection (NHWC vs NCHW)
def infer_layout(shape):
    # shape is (B, A, B2, C?) -> if last dim is small like 1,2,3 it's probably channels
    if len(shape) == 4:
        b, d1, d2, d3 = shape
        if d3 <= 8:   # tiny channels at end -> NHWC
            return "NHWC"
        if d1 <= 8:   # tiny channels second -> NCHW
            return "NCHW"
    return "UNKNOWN"

MODEL_LAYOUT = infer_layout(in_shape)
print("Inferred model layout:", MODEL_LAYOUT)

# ---------- Data loading ----------
def load_data():
    X = np.load(RF_INPUT_PATH)  # Expected (N, 1024, 1, 2) float32 (or compatible)
    if NUM_FRAMES is not None:
        X = X[:NUM_FRAMES]
    y = None
    if os.path.exists(RF_CLASSES_PATH):
        y = np.load(RF_CLASSES_PATH)
        y = y[:len(X)]
    snr = None
    if os.path.exists(RF_SNRS_PATH):
        snr = np.load(RF_SNRS_PATH)
        snr = snr[:len(X)]
    return X, y, snr

X, y_true, snr = load_data()
N = X.shape[0]
print(f"Loaded X: {X.shape} | y: {None if y_true is None else y_true.shape} | snr: {None if snr is None else snr.shape}")

# ---------- Preprocess & layout adapt ----------
# DPU typically expects int8 or float32 depending on compilation; the runner will accept numpy buffers directly.
# We'll feed float32 if your .xmodel expects float, else int8. If unsure, float32 is generally safe for F32 models.
# If your model is INT8-quantized, Vitis-AI DPU usually expects int8 inputs; you can cast below by toggling DTYPE_IN.
DTYPE_IN  = np.float32  # change to np.int8 if your xmodel is int8 input
DTYPE_OUT = np.float32  # we'll pull outputs into float32 for softmax

def to_model_layout(batch_nhwc):
    # batch_nhwc: (B, H, W, C) assumed
    if MODEL_LAYOUT == "NHWC":
        return batch_nhwc
    elif MODEL_LAYOUT == "NCHW":
        return np.transpose(batch_nhwc, (0, 3, 1, 2))
    else:
        # Fallback: try to match dims ignoring batch
        _, H, W, C = batch_nhwc.shape
        b, a1, a2, a3 = in_shape
        # Try to map to (B,C,H,W) if that matches more closely
        if C == a1 and H == a2 and W == a3:
            return np.transpose(batch_nhwc, (0, 3, 1, 2))
        return batch_nhwc

# Our stored X is likely (N, 1024, 1, 2). Treat that as NHWC already.
# If your data is in another order, adapt here before batching.

# ---------- Softmax ----------
def softmax_stable(x):
    x = x.astype(np.float32)
    m = np.max(x, axis=1, keepdims=True)
    np.subtract(x, m, out=x)
    np.exp(x, out=x)
    s = np.sum(x, axis=1, keepdims=True)
    return x / (s + 1e-12)

# ---------- Inference loop ----------
def run_dpu(X, batch_size=32):
    num_classes = out_shape[-1] if len(out_shape) == 2 else np.prod(out_shape[1:])
    preds = np.empty((len(X), num_classes), dtype=DTYPE_OUT)
    t0 = time.time()
    i = 0
    while i < len(X):
        j = min(i + batch_size, len(X))
        batch = X[i:j].astype(DTYPE_IN, copy=False)

        # Ensure NHWC for preprocessing; then convert to model layout
        if batch.ndim != 4:
            raise ValueError(f"Expected 4D input, got {batch.shape}")
        batch_m = to_model_layout(batch)

        # Allocate I/O buffers as lists per runner API
        # Input buffer must match runner's input dims for the current batch size.
        # Some runners insist the first dim equals the compiled batch; often 1 is OK but we handle dynamic batch by slicing.
        # Create buffers with the runner's expected shape, then copy batch into the beginning.
        bsz = batch_m.shape[0]
        in_buf  = [np.empty((in_shape[0], *in_shape[1:]), dtype=DTYPE_IN)]
        out_buf = [np.empty((out_shape[0], *out_shape[1:]), dtype=np.float32)]

        # Copy our data into the front of the input buffer
        in_view = in_buf[0][:bsz]
        if in_view.shape != batch_m.shape:
            # If compiled batch != runtime batch, we copy only a slice
            in_view[...] = batch_m
        else:
            in_buf[0][...] = batch_m

        # Execute
        jid = dpu.execute_async(in_buf, out_buf)
        dpu.wait(jid)

        # Read only the valid front slice
        raw = out_buf[0][:bsz]
        # Flatten to (B, C) if needed
        logits = raw.reshape(bsz, -1).astype(np.float32, copy=False)
        preds[i:j] = logits
        i = j
    dt = time.time() - t0
    ips = len(X) / dt if dt > 0 else float('inf')
    print(f"Inference done: {len(X)} frames in {dt:.3f}s  ({ips:.1f} FPS)")
    return preds

logits = run_dpu(X, BATCH_SIZE)
probs  = softmax_stable(logits)
y_pred = np.argmax(probs, axis=1)

# ---------- Accuracy ----------
if y_true is not None:
    acc = (y_pred == y_true).mean()
    print(f"Top-1 Accuracy: {acc*100:.2f}%  ({y_pred.size} samples)")
else:
    print("No ground-truth labels found; skipping accuracy.")

# ---------- Confusion Matrix ----------
if y_true is not None and HAVE_SK:
    cm = confusion_matrix(y_true, y_pred, labels=np.arange(len(MODS)))
    # Normalize by true class (row)
    cmn = cm.astype(np.float32) / (cm.sum(axis=1, keepdims=True) + 1e-12)

    def plot_confusion_matrix(cm_norm, labels, title="Normalized Confusion Matrix"):
        plt.figure(figsize=(8, 6))
        plt.imshow(cm_norm, interpolation='nearest', cmap=plt.cm.Blues)
        plt.title(title)
        plt.colorbar()
        ticks = np.arange(len(labels))
        plt.xticks(ticks, labels, rotation=45, ha='right')
        plt.yticks(ticks, labels)
        thresh = cm_norm.max() / 2.0
        for i in range(cm_norm.shape[0]):
            for j in range(cm_norm.shape[1]):
                plt.text(j, i, f"{cm_norm[i, j]*100:.1f}%",
                         ha="center", va="center",
                         color="white" if cm_norm[i, j] > thresh else "black", fontsize=8)
        plt.ylabel("True")
        plt.xlabel("Predicted")
        plt.tight_layout()

    plot_confusion_matrix(cmn, MODS)
    plt.show()
elif y_true is not None and not HAVE_SK:
    print("Install scikit-learn to plot the confusion matrix: pip install scikit-learn")

# ---------- Accuracy vs SNR ----------
if y_true is not None and snr is not None:
    uniq = np.unique(snr)
    acc_per = []
    for s in uniq:
        idx = np.where(snr == s)[0]
        if idx.size == 0: 
            acc_per.append(np.nan)
            continue
        acc_s = (y_pred[idx] == y_true[idx]).mean()
        acc_per.append(acc_s)
    plt.figure(figsize=(8, 4))
    plt.plot(uniq, acc_per, 'o-', label="Accuracy")
    plt.grid(True)
    plt.xlabel("SNR")
    plt.ylabel("Accuracy")
    plt.title("Accuracy vs SNR (DPU Inference)")
    plt.ylim(0, 1.0)
    plt.legend()
    plt.show()
else:
    print("SNR array not found or no labels; skipping Acc vs SNR.")

# ---------- Cleanup ----------
del dpu
del overlay
